# Thai Call Center ASR Submission Notebook

This notebook adapts the reference Thai Whisper ASR pipeline for `dataset/individual-test-thai-call-center-asr`.

Technique used:
- read the local Kaggle-style sample submission;
- optionally denoise audio with spectral subtraction and wavelet thresholding;
- transcribe Thai speech with a Hugging Face Whisper ASR pipeline;
- preserve sample-submission order and write `file_name,text` output.

Run the quick validation cells first, then set `RUN_FULL_BATCH = True` for the full dataset.

## 1. Optional dependency install

Uncomment the install line if the runtime does not already have the ASR/audio packages.

In [ ]:
# %pip install -q transformers accelerate librosa soundfile noisereduce PyWavelets tqdm pandas torch

## 2. Imports and paths

In [ ]:
from __future__ import annotations

import gc
import os
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from transformers import pipeline as hf_pipeline

try:
    import librosa
    import noisereduce as nr
    import pywt
    import soundfile as sf
except ImportError as exc:
    raise ImportError(
        "Missing audio dependencies. Run the optional install cell, then restart the kernel."
    ) from exc

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "call_center_asr":
    PROJECT_ROOT = PROJECT_ROOT.parents[1]

DATASET_DIR = PROJECT_ROOT / "dataset" / "individual-test-thai-call-center-asr"
AUDIO_DIR = DATASET_DIR / "audio_final" / "audio"
SAMPLE_SUBMISSION = DATASET_DIR / "sample_submission.csv"
OUTPUT_CSV = PROJECT_ROOT / "submission_call_center_asr.csv"

print("Project root:", PROJECT_ROOT)
print("Dataset dir:", DATASET_DIR)
print("Audio dir:", AUDIO_DIR)
print("Sample submission:", SAMPLE_SUBMISSION)

## 3. Configuration

Use `MAX_FILES` for a quick smoke test. Set it to `None` only after the first few files run correctly.

In [ ]:
ASR_MODEL_ID = os.getenv("ASR_MODEL_ID", "biodatlab/whisper-th-medium-combined")
LANGUAGE = "th"
TASK = "transcribe"
CHUNK_LENGTH_S = 20
BATCH_SIZE = 16

ENABLE_DENOISE = False
DENOISE_PROP_DECREASE = 0.8
WAVELET = "db4"
WAVELET_LEVEL = 1
TARGET_SR = 16000

MAX_FILES = 5  # set to None to process all rows in smoke-test mode
RUN_FULL_BATCH = False

DEVICE = 0 if torch.cuda.is_available() else -1
TORCH_DTYPE = torch.float16 if torch.cuda.is_available() else torch.float32

print("Model:", ASR_MODEL_ID)
print("Device:", "cuda:0" if DEVICE == 0 else "cpu")
print("Torch dtype:", TORCH_DTYPE)
print("Denoise:", ENABLE_DENOISE)

## 4. Inspect dataset

In [ ]:
sample_df = pd.read_csv(SAMPLE_SUBMISSION)
assert list(sample_df.columns) == ["file_name", "text"], sample_df.columns.tolist()
assert AUDIO_DIR.exists(), AUDIO_DIR

available_audio = {p.name for p in AUDIO_DIR.glob("*.wav")}
missing = sorted(set(sample_df["file_name"]) - available_audio)

print("Submission rows:", len(sample_df))
print("Audio files:", len(available_audio))
print("Missing referenced files:", len(missing))
display(sample_df.head())
if missing[:5]:
    print("Missing examples:", missing[:5])

## 5. Denoising helpers

This mirrors the reference notebook's optional two-stage denoise: spectral subtraction followed by wavelet soft thresholding.

In [ ]:
def wavelet_denoise(data: np.ndarray, wavelet: str = WAVELET, level: int = WAVELET_LEVEL) -> np.ndarray:
    coeff = pywt.wavedec(data, wavelet, mode="per")
    detail_index = -level
    sigma = np.median(np.abs(coeff[detail_index])) / 0.6745
    threshold = sigma * np.sqrt(2 * np.log(max(len(data), 2))) * 0.3

    for i in range(1, len(coeff)):
        if i >= level:
            coeff[i] = pywt.threshold(coeff[i], value=threshold, mode="soft")

    reconstructed = pywt.waverec(coeff, wavelet, mode="per")
    return reconstructed[: len(data)].astype(np.float32, copy=False)


def preprocess_audio(audio_path: Path, temp_dir: Path) -> str:
    if not ENABLE_DENOISE:
        return str(audio_path)

    y, sr = librosa.load(audio_path, sr=TARGET_SR, mono=True)
    y_nr = nr.reduce_noise(y=y, sr=sr, prop_decrease=DENOISE_PROP_DECREASE)
    y_dn = wavelet_denoise(y_nr)

    temp_path = temp_dir / f"{audio_path.stem}_denoised.wav"
    sf.write(temp_path, y_dn, TARGET_SR)
    return str(temp_path)

## 6. Load ASR model

In [ ]:
asr_pipe = hf_pipeline(
    task="automatic-speech-recognition",
    model=ASR_MODEL_ID,
    chunk_length_s=CHUNK_LENGTH_S,
    torch_dtype=TORCH_DTYPE,
    device=DEVICE,
    batch_size=BATCH_SIZE,
)

try:
    asr_pipe.model.config.forced_decoder_ids = asr_pipe.tokenizer.get_decoder_prompt_ids(
        language=LANGUAGE,
        task=TASK,
    )
    GENERATE_KWARGS = None
except Exception:
    GENERATE_KWARGS = {"language": "<|th|>", "task": TASK}

print("ASR pipeline loaded")

## 7. Single-file smoke test

In [ ]:
def transcribe_one(file_name: str, temp_dir: Path) -> str:
    audio_path = AUDIO_DIR / file_name
    if not audio_path.exists():
        raise FileNotFoundError(audio_path)

    model_input = preprocess_audio(audio_path, temp_dir)
    kwargs = {"return_timestamps": False}
    if GENERATE_KWARGS is not None:
        kwargs["generate_kwargs"] = GENERATE_KWARGS

    result = asr_pipe(model_input, **kwargs)
    return str(result.get("text", "")).strip()

with tempfile.TemporaryDirectory() as tmp:
    first_file = sample_df.loc[0, "file_name"]
    first_text = transcribe_one(first_file, Path(tmp))

print(first_file)
print(first_text)

## 8. Batch transcription

Keep `RUN_FULL_BATCH = False` until the smoke test output looks reasonable.

In [ ]:
run_df = sample_df.copy()
if not RUN_FULL_BATCH and MAX_FILES is not None:
    run_df = run_df.head(MAX_FILES)

results = []
errors = []

with tempfile.TemporaryDirectory() as tmp:
    temp_dir = Path(tmp)
    for file_name in tqdm(run_df["file_name"], total=len(run_df), desc="Transcribing"):
        try:
            text = transcribe_one(file_name, temp_dir)
        except Exception as exc:
            text = ""
            errors.append({"file_name": file_name, "error": repr(exc)})
        results.append({"file_name": file_name, "text": text})

pred_df = pd.DataFrame(results)
print("Rows transcribed:", len(pred_df))
print("Errors:", len(errors))
display(pred_df.head())
if errors:
    display(pd.DataFrame(errors).head(20))

## 9. Write submission CSV

For smoke tests, only the transcribed subset is filled and the rest stays blank. For final submission, set `RUN_FULL_BATCH = True` and run all rows.

In [ ]:
submission_df = sample_df.copy()
submission_df["text"] = ""
submission_df = submission_df.drop(columns=["text"]).merge(pred_df, on="file_name", how="left")
submission_df["text"] = submission_df["text"].fillna("")
submission_df = submission_df[["file_name", "text"]]

submission_df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8")
print("Wrote:", OUTPUT_CSV)
print("Rows:", len(submission_df))
display(submission_df.head())

## 10. Cleanup GPU memory

In [ ]:
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()